In [6]:
import cv2
import numpy as np
import os

input_folder = "enhanced_frames"
output_folder = "output_combined"

os.makedirs(output_folder, exist_ok=True)

def detect_pothole(image, output):

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (7,7), 0)

    mask = cv2.adaptiveThreshold(
        blur, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        21, 5
    )

    kernel_line = np.ones((1, 9), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_line)

    kernel = np.ones((5,5), np.uint8)
    clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    kernel_merge = np.ones((15,15), np.uint8)
    merged = cv2.morphologyEx(clean, cv2.MORPH_CLOSE, kernel_merge, iterations=2)

    contours, _ = cv2.findContours(merged, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    count = 0

    for cnt in contours:

        area = cv2.contourArea(cnt)
        if area < 1500:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        aspect_ratio = w / (h + 1e-5)
        if aspect_ratio > 8 or aspect_ratio < 0.15:
            continue

        roi = gray[y:y+h, x:x+w]
        if np.mean(roi) > 230:
            continue

        cv2.rectangle(output, (x,y), (x+w,y+h), (0,0,255), 3)
        cv2.putText(output, "Pothole", (x, y-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)

        count += 1

    return count


def detect_alligator(gray, output):

    h, w = gray.shape

    mask = np.zeros((h, w), dtype=np.uint8)
    mask[int(h*0.40):h, int(w*0.15):int(w*0.85)] = 255

    roi = cv2.bitwise_and(gray, gray, mask=mask)

    blur = cv2.GaussianBlur(roi, (7,7), 0)
    edges = cv2.Canny(blur, 50, 140)

    block_size = 50
    step = 25
    crack_map = np.zeros_like(edges)

    for y in range(0, h - block_size, step):
        for x in range(0, w - block_size, step):

            region = edges[y:y+block_size, x:x+block_size]
            if np.mean(region > 0) > 0.18:
                crack_map[y:y+block_size, x:x+block_size] = 255

    kernel = np.ones((21,21), np.uint8)
    crack_map = cv2.morphologyEx(crack_map, cv2.MORPH_CLOSE, kernel, 2)

    contours, _ = cv2.findContours(crack_map, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    count = 0

    for c in contours:

        area = cv2.contourArea(c)
        if area < 4000:
            continue

        x, y, w, h = cv2.boundingRect(c)

        roi_local = gray[y:y+h, x:x+w]
        if np.std(roi_local) < 15 or np.mean(roi_local) > 235:
            continue

        cv2.rectangle(output, (x,y), (x+w,y+h), (0,255,0), 3)
        cv2.putText(output, "Alligator Crack", (x, y-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

        count += 1

    return count


def detect_edge_cracks(gray, output):

    h, w = gray.shape

    mask = np.zeros((h, w), dtype=np.uint8)
    mask[int(h*0.45):h, :int(w*0.25)] = 255
    mask[int(h*0.45):h, int(w*0.75):] = 255

    roi = cv2.bitwise_and(gray, gray, mask=mask)

    blur = cv2.GaussianBlur(roi, (5,5), 0)
    edges = cv2.Canny(blur, 40, 120)

    crack_map = np.zeros_like(edges)

    block_w, block_h = 40, 60

    for y in range(0, h - block_h, 20):
        for x in range(0, w - block_w, 20):

            region = edges[y:y+block_h, x:x+block_w]

            if np.mean(region > 0) > 0.08:
                crack_map[y:y+block_h, x:x+block_w] = 255

    kernel = np.ones((15,15), np.uint8)
    crack_map = cv2.morphologyEx(crack_map, cv2.MORPH_CLOSE, kernel, 2)

    contours, _ = cv2.findContours(crack_map, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    count = 0

    for c in contours:

        area = cv2.contourArea(c)
        if area < 2000:
            continue

        x, y, w, h = cv2.boundingRect(c)

        if h / (w + 1e-5) < 1.2:
            continue

        cv2.rectangle(output, (x,y), (x+w,y+h), (255,255,0), 2)
        cv2.putText(output, "Edge Crack", (x, y-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)

        count += 1

    return count


def detect_faded_crossing(frame, output):

    h, w = frame.shape[:2]

    mask = np.zeros((h, w), dtype=np.uint8)
    mask[int(h*0.45):h, int(w*0.05):int(w*0.95)] = 255

    roi = frame[int(h*0.45):h, :]
    roi_mask = mask[int(h*0.45):h, :]

    roi = cv2.bitwise_and(roi, roi, mask=roi_mask)

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

    clahe = cv2.createCLAHE(2.5, (8,8))
    enhanced = clahe.apply(gray)

    blur = cv2.GaussianBlur(enhanced, (5,5), 0)

    _, binary = cv2.threshold(blur, 130, 255, cv2.THRESH_BINARY)

    kernel = np.ones((5,25), np.uint8)
    morph = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, 2)

    contours, _ = cv2.findContours(morph, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    count = 0

    for c in contours:

        area = cv2.contourArea(c)
        if area < 800:
            continue

        x, y, w, h = cv2.boundingRect(c)

        if w / (h + 1e-5) < 1.8:
            continue

        cv2.rectangle(output,
                      (x, y + int(frame.shape[0]*0.45)),
                      (x+w, y+h + int(frame.shape[0]*0.45)),
                      (0,165,255), 2)

        count += 1

    return count



files = sorted(os.listdir(input_folder))

for file in files:

    if not file.lower().endswith(('.png','.jpg','.jpeg')):
        continue

    path = os.path.join(input_folder, file)
    frame = cv2.imread(path)

    if frame is None:
        continue

    frame = cv2.resize(frame, (640, 480))
    output = frame.copy()

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    p = detect_pothole(frame, output)
    a = detect_alligator(gray, output)
    e = detect_edge_cracks(gray, output)
    c = detect_faded_crossing(frame, output)

    # summary
    cv2.putText(output, f"P:{p} A:{a} E:{e} C:{c}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255,255,255),
                2)

    cv2.imwrite(os.path.join(output_folder, file), output)

    print(f"{file} → P:{p}, A:{a}, E:{e}, C:{c}")

print("\n ALL DETECTION COMPLETE (UNIFIED SYSTEM)")

AlligatorCracking_frame_0.jpg → P:4, A:1, E:2, C:1
AlligatorCracking_frame_20.jpg → P:4, A:1, E:2, C:1
AlligatorCracking_frame_35.jpg → P:5, A:1, E:2, C:1
AlligatorCracking_frame_4.jpg → P:2, A:1, E:2, C:1
AlligatorCracking_frame_51.jpg → P:3, A:1, E:2, C:1
AlligatorCracking_frame_9.jpg → P:5, A:1, E:2, C:1
EdgeCracking3_frame_72.jpg → P:3, A:1, E:2, C:1
EdgeCracking4_frame_123.jpg → P:0, A:1, E:2, C:1
EdgeCracking4_frame_129.jpg → P:0, A:1, E:2, C:1
EdgeCracking4_frame_130.jpg → P:0, A:1, E:2, C:1
EdgeCracking4_frame_133.jpg → P:0, A:1, E:2, C:1
EdgeCracking4_frame_144.jpg → P:0, A:1, E:2, C:1
EdgeCracking4_frame_154.jpg → P:0, A:1, E:2, C:1
EdgeCracking9_frame_185.jpg → P:0, A:1, E:2, C:1
EdgeCracking9_frame_202.jpg → P:0, A:1, E:2, C:1
EdgeCracking9_frame_206.jpg → P:0, A:2, E:2, C:1
EdgeCracking9_frame_217.jpg → P:0, A:1, E:2, C:1
RoadPotholes2_frame_522.jpg → P:6, A:1, E:2, C:1
RoadPotholes3_frame_575.jpg → P:1, A:1, E:2, C:1
RoadPotholes3_frame_578.jpg → P:1, A:1, E:2, C:1
RoadPo